# 01 - Parse HTML Dictionary Data

This notebook parses the English-Hiligaynon-Akeanon dictionary from the HTML source file
into structured JSON and CSV formats.

**Source:** [How To Speak Aklanon The Easy Way](https://howtospeakaklanon.blogspot.com/2011/09/english-hiligaynon-akeanon.html)  
**Author:** Melchor F. Cichon

In [7]:
import sys
from pathlib import Path

# Ensure project root is on the path
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

Project root: c:\Users\gioan\Documents\GitHub\chatbutt


In [8]:
# Reload modules after code changes
import importlib
import src.parse_html
importlib.reload(src.parse_html)
from src.parse_html import load_html, extract_raw_lines, parse_html_dictionary, save_json, save_csv, create_trainset, create_devset

html_path = project_root / 'data' / 'raw' / 'english-hiligaynon-akeanon.html'
if not html_path.exists():
    html_path = project_root / 'english-hiligaynon-akeanon.html'

entries = parse_html_dictionary(html_path)
print(f'\nTotal parsed entries: {len(entries)}')

Parsed 10897 entries, skipped 0 lines

Total parsed entries: 10897


In [9]:
import pandas as pd
df = pd.DataFrame(entries)
print(f'Total entries: {len(df)}')
print(f'Entries with Hiligaynon: {df["hiligaynon"].apply(len).gt(0).sum()}')
print(f'Entries with Akeanon: {df["akeanon"].apply(len).gt(0).sum()}')
print(f'Entries with both: {(df["hiligaynon"].apply(len).gt(0) & df["akeanon"].apply(len).gt(0)).sum()}')
print(f'Unique English words: {df["english"].nunique()}')
print(f'\nSample entries with all three languages:')
both = df[(df["hiligaynon"].apply(len) > 0) & (df["akeanon"].apply(len) > 0)]
for _, row in both.head(10).iterrows():
    print(f'  EN: {row["english"]:30s} HIL: {str(row["hiligaynon"]):40s} AKE: {str(row["akeanon"])}')

Total entries: 10897
Entries with Hiligaynon: 10846
Entries with Akeanon: 766
Entries with both: 763
Unique English words: 7661

Sample entries with all three languages:
  EN: a                              HIL: ['isa']                                  AKE: ['saea']
  EN: aback adverb ( to be taken aback) HIL: ['palak']                                AKE: ['paeak']
  EN: abandon                        HIL: ['pabayaan', 'abandonar']                AKE: ['abandonar', 'aywan', 'pabay-an']
  EN: abandoned                      HIL: ['sim-ong']                              AKE: ['-ginpabay-an', 'gin-aywan', 'gin-abandonar']
  EN: abatoir                        HIL: ['ihawan']                               AKE: ['matansahan']
  EN: abbreviation                   HIL: ['lip-ot']                               AKE: ['matag-od']
  EN: abc                            HIL: ['abakada']                              AKE: ['abakada']
  EN: abdomen                        HIL: ['tinai', 'tiyan']          

In [10]:
# Save parsed data
processed_dir = project_root / 'data' / 'processed'
from src.parse_html import save_json, save_csv
save_json(entries, processed_dir / 'dictionary.json')
save_csv(entries, processed_dir / 'dictionary.csv')

# Create train/dev splits
examples_dir = project_root / 'data' / 'examples'
trainset = create_trainset(entries, examples_dir / 'trainset.json', n_samples=50)
devset = create_devset(entries, examples_dir / 'devset.json', n_samples=25)
print(f'\nTrainset: {len(trainset)} examples, Devset: {len(devset)} examples')

Saved 10897 entries to c:\Users\gioan\Documents\GitHub\chatbutt\data\processed\dictionary.json
Saved 10897 entries to c:\Users\gioan\Documents\GitHub\chatbutt\data\processed\dictionary.csv
Saved 50 training examples to c:\Users\gioan\Documents\GitHub\chatbutt\data\examples\trainset.json
Saved 25 training examples to c:\Users\gioan\Documents\GitHub\chatbutt\data\examples\devset.json

Trainset: 50 examples, Devset: 25 examples


In [11]:
# Build ChromaDB vector index
from src.retriever import build_index, retrieve

dict_path = processed_dir / 'dictionary.json'
chroma_dir = project_root / 'chroma_db'

collection = build_index(dict_path, chroma_dir)

C:\Users\gioan\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:28<00:00, 2.92MiB/s]  


Indexed 10897 dictionary entries into ChromaDB at c:\Users\gioan\Documents\GitHub\chatbutt\chroma_db


In [12]:
# Test retrieval
test_queries = ['hello', 'good morning', 'love', 'eat', 'water', 'house', 'friend', 'beautiful']
for q in test_queries:
    results = retrieve(q, collection, chroma_dir, top_k=3)
    print(f"\nQuery: '{q}'")
    for r in results:
        print(f'  -> {r}')


Query: 'hello'
  -> English: greetings | Hiligaynon: pagtamyaw
  -> English: hey | Hiligaynon: hoy
  -> English: welcome | Hiligaynon: party bienvenida

Query: 'good morning'
  -> English: morning | Hiligaynon: aga
  -> English: morning | Hiligaynon: kaagahon
  -> English: morning | Hiligaynon: star kabugwason

Query: 'love'
  -> English: love-vine | Hiligaynon: kadena de amor
  -> English: love | (noun) | Hiligaynon: gugma
  -> English: love | (verb) | Hiligaynon: higugma

Query: 'eat'
  -> English: eat | Hiligaynon: ( gulp down without chewing) lamon
  -> English: eat | Hiligaynon: raw fish or meat kilaw
  -> English: eating | Hiligaynon: place karinderiya, kalan-an

Query: 'water'
  -> English: water | Hiligaynon: jar banga
  -> English: water | (noun) | Hiligaynon: tubig, agwa
  -> English: pool | Hiligaynon: ( puddle) danaw

Query: 'house'
  -> English: house | Hiligaynon: balay
  -> English: house | Hiligaynon: owner tagbalay
  -> English: home | Hiligaynon: ( house) balay

Quer

## Step 1: Load and inspect the HTML file

In [2]:
from src.parse_html import load_html, extract_raw_lines

# Try data/raw first, fall back to project root
html_path = project_root / 'data' / 'raw' / 'english-hiligaynon-akeanon.html'
if not html_path.exists():
    html_path = project_root / 'english-hiligaynon-akeanon.html'

print(f'Loading HTML from: {html_path}')
html_content = load_html(html_path)
print(f'HTML file size: {len(html_content):,} characters')

Loading HTML from: c:\Users\gioan\Documents\GitHub\chatbutt\data\raw\english-hiligaynon-akeanon.html
HTML file size: 389,294 characters


In [3]:
# Extract raw lines from the HTML
raw_lines = extract_raw_lines(html_content)
print(f'Extracted {len(raw_lines)} raw lines')
print('\nFirst 20 lines:')
for i, line in enumerate(raw_lines[:20]):
    print(f'  {i+1:4d}: {line}')

Extracted 10897 raw lines

First 20 lines:
     1: a ( indefinite article)-- isa--saea
     2: aback adverb ( to be taken aback)-- palak--paeak
     3: abandon verb -- pabayaan , abandonar--abandonar, aywan, pabay-an
     4: abandoned adjective -- sim-ong---ginpabay-an, gin-aywan, gin-abandonar
     5: abatoir noun -- ihawan--matansahan
     6: abbreviation noun -- lip-ot--matag-od
     7: ABC -- abakada--abakada
     8: abdomen noun --tinai , see also: tiyan--tinai, tiyan
     9: abduct verb --agaw--agaw
    10: abeyance -- paghulat--paghueat
    11: abide verb --puyo , istar--istar, tiner
    12: ability noun -- abilidad , kaayo , pagkabatid--abilidad, kaamad, pagkalisto
    13: able ( adjective)--makasangkul--makasangkoe; makasarang
    14: able to ` ( can do it) sarang , ako--sarang, masarangan
    15: abnormal adjective --abnormal , tuhay sa kinaandan--abnormal
    16: aboard -- sakay--sakay
    17: aboard ( adverb)--nakalulan--nakasakay
    18: abolish ( verb)--kuhaon , kuhaan si

In [ ]:
# Look at some lines from the middle and end too
print('Lines 500-510:')
for i, line in enumerate(raw_lines[499:510], start=500):
    print(f'  {i:4d}: {line}')

print(f'\nLast 10 lines:')
for i, line in enumerate(raw_lines[-10:], start=len(raw_lines)-9):
    print(f'  {i:4d}: {line}')

## Step 2: Parse entries into structured format

In [4]:
from src.parse_html import parse_html_dictionary

entries = parse_html_dictionary(html_path)
print(f'\nTotal parsed entries: {len(entries)}')

Parsed 10852 entries, skipped 45 lines
Sample skipped lines:
  - bankruptcykaputohan
  - commercialkomersyal
  - compulsorykompulsaryo
  - conveyancesalakyan
  - corporationkorporasyon

Total parsed entries: 10852


In [5]:
# Inspect a few entries
import json

print('Sample entries:')
for entry in entries[:10]:
    print(json.dumps(entry, ensure_ascii=False, indent=2))
    print()

Sample entries:
{
  "english": "a",
  "pos": "indefinite article",
  "hiligaynon": [
    "isa"
  ],
  "akeanon": [
    "saea"
  ],
  "raw_line": "a ( indefinite article)-- isa--saea"
}

{
  "english": "aback adverb ( to be taken aback)",
  "pos": "",
  "hiligaynon": [
    "palak"
  ],
  "akeanon": [
    "paeak"
  ],
  "raw_line": "aback adverb ( to be taken aback)-- palak--paeak"
}

{
  "english": "abandon",
  "pos": "verb",
  "hiligaynon": [
    "pabayaan",
    "abandonar"
  ],
  "akeanon": [
    "abandonar",
    "aywan",
    "pabay-an"
  ],
  "raw_line": "abandon verb -- pabayaan , abandonar--abandonar, aywan, pabay-an"
}

{
  "english": "abandoned",
  "pos": "adjective",
  "hiligaynon": [
    "sim-ong"
  ],
  "akeanon": [
    "-ginpabay-an",
    "gin-aywan",
    "gin-abandonar"
  ],
  "raw_line": "abandoned adjective -- sim-ong---ginpabay-an, gin-aywan, gin-abandonar"
}

{
  "english": "abatoir",
  "pos": "noun",
  "hiligaynon": [
    "ihawan"
  ],
  "akeanon": [
    "matansahan"
  

In [6]:
# Statistics
import pandas as pd

df = pd.DataFrame(entries)
print(f'Total entries: {len(df)}')
print(f'\nEntries with Hiligaynon translations: {df["hiligaynon"].apply(len).gt(0).sum()}')
print(f'Entries with Akeanon translations: {df["akeanon"].apply(len).gt(0).sum()}')
print(f'Entries with both translations: {(df["hiligaynon"].apply(len).gt(0) & df["akeanon"].apply(len).gt(0)).sum()}')
print(f'\nParts of speech distribution:')
print(df['pos'].value_counts().head(15))
print(f'\nUnique English words: {df["english"].nunique()}')

Total entries: 10852

Entries with Hiligaynon translations: 10835
Entries with Akeanon translations: 132
Entries with both translations: 132

Parts of speech distribution:
pos
                       10517
noun                     153
verb                      89
adjective                 56
adverb                    23
preposition                6
interjection               3
indefinite article         1
adverb, preposition        1
verb to be                 1
singular                   1
plural                     1
Name: count, dtype: int64

Unique English words: 7737


## Step 3: Save parsed data

In [ ]:
from src.parse_html import save_json, save_csv

processed_dir = project_root / 'data' / 'processed'

save_json(entries, processed_dir / 'dictionary.json')
save_csv(entries, processed_dir / 'dictionary.csv')

## Step 4: Create train/dev splits for DSPy optimization

In [ ]:
from src.parse_html import create_trainset, create_devset

examples_dir = project_root / 'data' / 'examples'

trainset = create_trainset(entries, examples_dir / 'trainset.json', n_samples=50)
devset = create_devset(entries, examples_dir / 'devset.json', n_samples=25)

print(f'\nTrainset sample:')
for ex in trainset[:3]:
    print(json.dumps(ex, ensure_ascii=False, indent=2))

## Step 5: Build the ChromaDB vector index

In [ ]:
from src.retriever import build_index, retrieve

dict_path = processed_dir / 'dictionary.json'
chroma_dir = project_root / 'chroma_db'

collection = build_index(dict_path, chroma_dir)

In [ ]:
# Test retrieval
test_queries = ['hello', 'good morning', 'love', 'eat', 'water', 'house', 'friend']

for q in test_queries:
    results = retrieve(q, collection, chroma_dir, top_k=3)
    print(f"\nQuery: '{q}'")
    for r in results:
        print(f'  -> {r}')

---
**Next:** Go to `02_prototype.ipynb` to test the DSPy translation pipeline.